# 1. Setup

In [1]:
# @title
import pandas as pd
import json
import requests
import time
import holidays
from datetime import datetime, timezone, timedelta
from math import radians, sin, cos, sqrt, atan2
from sqlalchemy import create_engine, text
from google.colab import userdata

In [2]:
# @title
TOMTOM_API_KEY = userdata.get('TOMTOM_API_KEY')
DB_URL = userdata.get('SUPABASE_DB_URL')

INTERVAL_MINUTES = 30          # Collect every 30 minutes
MAX_CALLS_PER_RUN = None       # Set to number for testing (e.g., 20), None for unlimited
DELAY_BETWEEN_API_CALLS = 0.5  # seconds between API calls (be polite)

# Vietnam timezone (UTC+7)
VIETNAM_TZ = timezone(timedelta(hours=7))

HANOI_LAT = 21.0285
HANOI_LON = 105.8542

engine = create_engine(DB_URL)

routes_df = pd.read_sql("""
    SELECT route_id, route_name, start_lat, start_lon, end_lat, end_lon
    FROM routes
    ORDER BY route_id
""", engine)
routes_df['route_id'] = routes_df['route_id'].astype('Int64')

# 2. Traffic Data Collection
## 2.1 TomTom API Functions

In [3]:
# @title
def get_route_travel_time(api_key, start_lat, start_lon, end_lat, end_lon):
    """
    Call TomTom Routing API to get travel time with traffic.

    Returns:
        tuple: (travel_time_seconds, no_traffic_time_seconds, historic_time_seconds, status)
    """

    # Construct origin and destination strings
    origin = f"{start_lat},{start_lon}"
    destination = f"{end_lat},{end_lon}"

    # API endpoint
    url = f"https://api.tomtom.com/routing/1/calculateRoute/{origin}:{destination}/json"

    # Parameters
    params = {
        "key": api_key,
        "traffic": "true",              # Include live traffic
        "computeTravelTimeFor": "all",  # Get traffic, no traffic, and historic times
        "travelMode": "car",
        "vehicleCommercial": "false"
    }

    try:
        response = requests.get(url, params=params, timeout=30)

        if response.status_code == 200:
            data = response.json()

            # Extract travel times from response
            summary = data['routes'][0]['summary']
            travel_time = summary.get('travelTimeInSeconds')
            no_traffic_time = summary.get('noTrafficTravelTimeInSeconds')
            historic_time = summary.get('historicTrafficTravelTimeInSeconds')

            return travel_time, no_traffic_time, historic_time, "SUCCESS"
        else:
            print(f"API Error {response.status_code}: {response.text[:100]}")
            return None, None, None, f"ERROR_{response.status_code}"

    except requests.exceptions.Timeout:
        print(f"Timeout error")
        return None, None, None, "TIMEOUT"
    except Exception as e:
        print(f"Exception: {str(e)[:100]}")
        return None, None, None, "EXCEPTION"

## 2.2 Database Functions

In [4]:
# @title
def insert_traffic_observation(conn, route_id, observed_time, travel_time_sec, no_traffic_sec):
    """Insert single traffic observation"""

    insert_query = """
    INSERT INTO traffic_observations (
        route_id, observed_time, travel_time_seconds, no_traffic_time_seconds
    ) VALUES (:route_id, :observed_time, :travel_time_seconds, :no_traffic_time_seconds)
    """

    conn.execute(text(insert_query), {
        'route_id': route_id,
        'observed_time': observed_time.isoformat(),
        'travel_time_seconds': travel_time_sec,
        'no_traffic_time_seconds': no_traffic_sec
    })
    conn.commit()

## 2.3 Main Collection Loop

In [ ]:
# @title
print("="*60)
print("STARTING TRAFFIC DATA COLLECTION")
print(f"Routes to collect: {len(routes_df)}")
print(f"Interval: {INTERVAL_MINUTES} minutes")

collection_count = 0

try:
    while True:
        timestamp = datetime.now(VIETNAM_TZ)
        print(f"\n{'='*60}")
        print(f"Collection started at: {timestamp.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"Collection #: {collection_count + 1}")
        print(f"{'='*60}")

        with engine.connect() as conn:

            # Collect data for all routes
            for idx, row in routes_df.iterrows():
                print(f"  [{idx+1:2d}/{len(routes_df)}] {row['route_name'][:20]}...",
                      end=" ", flush=True)

                travel, no_traffic, historic, status = get_route_travel_time(
                    TOMTOM_API_KEY,
                    row['start_lat'], row['start_lon'],
                    row['end_lat'], row['end_lon'],
                )

                if status == "SUCCESS" and travel:
                    insert_traffic_observation(
                        conn, row['route_id'], timestamp, travel, no_traffic
                    )

                    # Calculate minutes for display
                    travel_min = travel / 60
                    no_traffic_min = no_traffic / 60 if no_traffic else 0
                    ratio = travel / no_traffic if no_traffic else 0

                    if ratio < 1.3:
                        rate = "  x"
                    elif ratio < 2.0:
                        rate = " xx"
                    else:
                        rate = "xxx"

                    print(f"{rate} {travel_min:2.0f} min (ratio: {ratio:.2f})")
                else:
                    print(f"[ ERROR ] {status}")

                # Small delay between API calls to be polite
                time.sleep(DELAY_BETWEEN_API_CALLS)

        collection_count += 1

        # Check if max calls reached
        if MAX_CALLS_PER_RUN and collection_count >= MAX_CALLS_PER_RUN:
            print(f"\nReached maximum collections ({MAX_CALLS_PER_RUN}). Stopping.")
            break

        # Wait for next collection interval
        next_collection = timestamp + timedelta(minutes=INTERVAL_MINUTES)
        wait_seconds = INTERVAL_MINUTES * 60

        print(f"\nNext collection at: {next_collection.strftime('%H:%M:%S')}")
        print(f"Waiting {INTERVAL_MINUTES} minutes...\n")

        diff = datetime.now(VIETNAM_TZ) - timestamp
        time.sleep(wait_seconds - diff.seconds)

except KeyboardInterrupt:
    print(f"\n{'='*60}")
    print(f"Ending traffic data collection.")
    print(f"Number of collections: {collection_count}")
    print(f"{'='*60}")

STARTING TRAFFIC DATA COLLECTION
Routes to collect: 40
Interval: 30 minutes

Collection started at: 2026-05-22 07:38:41
Collection #: 1
  [ 1/40] Đường Quang Trung → ...  xx 26 min (ratio: 1.61)
  [ 2/40] Đường Trường Sa → Đư...   x  8 min (ratio: 1.15)
  [ 3/40] Phố Nguyễn Thế Rục →...  xx 32 min (ratio: 1.43)
  [ 4/40] Quốc lộ 1A cũ → Vòng...  xx 32 min (ratio: 1.48)
  [ 5/40] Cầu Nhật Tân → Đường...  xx 13 min (ratio: 1.58)
  [ 6/40] Quan La Sở → Đường V...   x 12 min (ratio: 1.12)
  [ 7/40] Đường Hoàng Quốc Việ...  xx 18 min (ratio: 1.68)
  [ 8/40] Đường Cầu Diễn → Cầu... xxx 14 min (ratio: 2.28)
  [ 9/40] Đường Xuân Thủy → Đư...  xx 15 min (ratio: 1.78)
  [10/40] Đường cao tốc Vành đ...  xx  5 min (ratio: 1.63)
  [11/40] Hầm chui Thanh Xuân ...  xx 26 min (ratio: 1.33)
  [12/40] Đường cao tốc Vành đ...   x 22 min (ratio: 1.14)
  [13/40] Đường Ngọc Hồi → Ngõ...  xx 20 min (ratio: 1.53)
  [14/40] Xã Phụng Công → Thổ ...   x 11 min (ratio: 1.25)
  [15/40] Hầm chui Cổ Linh → C...   x 

# 3. Weather Data
## 3.1 Open Meteo API Functions

In [ ]:
# @title
def get_weather_condition(weather_code):
    """Convert Open-Meteo weather code to human-readable condition."""
    if weather_code == 0:
        return 'Clear'
    elif weather_code in (1, 2, 3):
        return 'Clouds'
    elif weather_code in (45, 48):
        return 'Fog'
    elif weather_code in (51, 53, 55):
        return 'Drizzle'
    elif weather_code in (56, 57):
        return 'Freezing Drizzle'
    elif weather_code in (61, 63, 65):
        return 'Rain'
    elif weather_code in (66, 67):
        return 'Freezing Rain'
    elif weather_code in (71, 73, 75):
        return 'Snow'
    elif weather_code == 77:
        return 'Snow Grains'
    elif weather_code in (80, 81, 82):
        return 'Rain Showers'
    elif weather_code in (85, 86):
        return 'Snow Showers'
    elif weather_code == 95:
        return 'Thunderstorm'
    elif weather_code in (96, 99):
        return 'Thunderstorm with Hail'
    else:
        return 'Unknown'

def fetch_weather_for_date(target_date):
    """
    Fetch hourly weather data for Hanoi on a specific date.
    Returns list of dictionaries or empty list on failure.
    """
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": HANOI_LAT,
        "longitude": HANOI_LON,
        "hourly": [
            "temperature_2m",
            "relative_humidity_2m",
            "precipitation",
            "rain",
            "weather_code"
        ],
        "start_date": target_date.strftime("%Y-%m-%d"),
        "end_date": target_date.strftime("%Y-%m-%d"),
        "timezone": "Asia/Bangkok"
    }

    try:
        response = requests.get(url, params=params, timeout=30)
        if response.status_code == 200:
            data = response.json()
            hourly = data['hourly']

            weather_data = []
            for i in range(24):
                weather_code = hourly['weather_code'][i]
                weather_condition = get_weather_condition(weather_code)

                weather_data.append({
                    'observation_date': target_date,
                    'hour': i,
                    'temperature_celsius': hourly['temperature_2m'][i],
                    'humidity_percent': hourly['relative_humidity_2m'][i],
                    'precipitation_mm': hourly['precipitation'][i],
                    'rain_mm': hourly['rain'][i],
                    'weather_code': weather_code,
                    'weather_condition': weather_condition,
                    'source': 'Open-Meteo'
                })
            return weather_data
        else:
            print(f"  API Error {response.status_code}: {response.text[:100]}")
            return []
    except Exception as e:
        print(f"  Exception: {e}")
        return []

## 3.2 Database Functions

In [ ]:
# @title
def insert_weather_data(conn, weather_record):
    """
    Insert or update weather data for a specific date and hour.
    Uses ON CONFLICT to handle duplicates.
    """
    query = """
    INSERT INTO weather_data (
        observation_date, hour, temperature_celsius, humidity_percent,
        precipitation_mm, rain_mm, weather_code, weather_condition,
        source, created_at
    ) VALUES (
        :observation_date, :hour, :temperature_celsius, :humidity_percent,
        :precipitation_mm, :rain_mm, :weather_code, :weather_condition,
        :source, NOW()
    )
    ON CONFLICT (observation_date, hour)
    DO UPDATE SET
        temperature_celsius = EXCLUDED.temperature_celsius,
        humidity_percent = EXCLUDED.humidity_percent,
        precipitation_mm = EXCLUDED.precipitation_mm,
        rain_mm = EXCLUDED.rain_mm,
        weather_code = EXCLUDED.weather_code,
        weather_condition = EXCLUDED.weather_condition,
        source = EXCLUDED.source,
        created_at = NOW()
    """
    conn.execute(text(query), weather_record)
    conn.commit()

## 3.3 Helper Functions

In [ ]:
# @title
def populate_weather_for_date_range(start_date, end_date, delay_seconds=1):
    """
    Populate weather data for all dates in range [start_date, end_date].
    """
    current_date = start_date
    total_days = (end_date - start_date).days + 1
    success_count = 0

    print(f"Fetching weather data from {start_date} to {end_date} ({total_days} days)")
    print("="*60)

    with engine.connect() as conn:
        for i in range(total_days):
            date_str = current_date.strftime("%Y-%m-%d")
            print(f"[{i+1}/{total_days}] Fetching weather for {date_str}...", end=" ", flush=True)

            weather_data = fetch_weather_for_date(current_date)

            if weather_data:
                for record in weather_data:
                    insert_weather_data(conn, record)
                print(f"({len(weather_data)} hours)")
                success_count += 1
            else:
                print(f"Failed")

            # Be polite to the API
            time.sleep(delay_seconds)

            current_date += timedelta(days=1)

    print("="*60)
    print(f"Successfully populated {success_count}/{total_days} days")

def populate_weather_for_month(year, month):
    """Populate weather data for an entire month."""
    start_date = datetime(year, month, 1).date()

    # Get last day of the month
    if month == 12:
        end_date = datetime(year, month, 31).date()
    else:
        end_date = datetime(year, month + 1, 1).date() - timedelta(days=1)

    populate_weather_for_date_range(start_date, end_date)

## 3.4 Populate Weather Data

In [ ]:
# @title
print("="*60)
print("WEATHER DATA POPULATOR")
print("="*60)

date_range = pd.read_sql("""
    SELECT MIN(observed_time) as min_date, MAX(observed_time) as max_date
    FROM traffic_observations
""", engine)

min_date = pd.to_datetime(date_range['min_date'].iloc[0]).date()
max_date = pd.to_datetime(date_range['max_date'].iloc[0]).date()

print(f"Traffic data covers: {min_date} to {max_date}")

populate_weather_for_date_range(min_date, max_date)

WEATHER DATA POPULATOR
Traffic data covers: 2026-05-17 to 2026-05-21
Fetching weather data from 2026-05-17 to 2026-05-21 (5 days)
[1/5] Fetching weather for 2026-05-17... (24 hours)
[2/5] Fetching weather for 2026-05-18... (24 hours)
[3/5] Fetching weather for 2026-05-19... (24 hours)
[4/5] Fetching weather for 2026-05-20... (24 hours)
[5/5] Fetching weather for 2026-05-21... (24 hours)
Successfully populated 5/5 days


# 4. Holidays
## 4.1 Holidays Library Functions

In [ ]:
# @title
def get_holidays_from_library(year=2026):
    """Fetch Vietnamese holidays using the 'holidays' library."""

    all_holidays = []
    vn_holidays = holidays.Vietnam(years=year)

    for holiday_date, holiday_name in vn_holidays.items():
        all_holidays.append({
            'holiday_date': holiday_date,
            'holiday_name': holiday_name
        })

    return pd.DataFrame(all_holidays)

## 4.2 Database Functions

In [ ]:
# @title
def insert_holidays(df, engine):
    """Upload holidays to Supabase, avoiding duplicates."""

    # Ensure date format
    df['holiday_date'] = pd.to_datetime(df['holiday_date']).dt.date

    # Remove duplicates within the DataFrame
    df = df.drop_duplicates(subset=['holiday_date'])

    # Upload to Supabase
    try:
        df.to_sql('holidays', engine, if_exists='append', index=False)
        print("Holidays uploaded successfully")
    except Exception as e:
        if "duplicate key" in str(e).lower():
            df.to_sql('holidays', engine, if_exists='replace', index=False)
            print("Some holidays already exist. Holidays table replaced with new data.")
        else:
            print(f"Error: {e}")

## 4.3 Holidays Upload

In [ ]:
# @title
print("="*60)
print("VIETNAMESE HOLIDAYS UPLOADER")
print("="*60)

# Fetch holidays
print("\nFetching holidays from library...")
try:
    df_holidays = get_holidays_from_library(year=2026)
    insert_holidays(df_holidays, engine)
    print(f"   Found {len(df_holidays)} holidays from library")
except Exception as e:
    print(f"   Library failed: {e}")

VIETNAMESE HOLIDAYS UPLOADER

Fetching holidays from library...
Holidays uploaded successfully
   Found 12 holidays from library


# 5. Route Name Update (OpenStreetMap Nominatim)
## 5.1 Nominatim API Functions

In [ ]:
# @title
def get_location_name(lat, lon, max_length=40):
    """Get shorter location name."""
    url = f"https://nominatim.openstreetmap.org/reverse?lat={lat}&lon={lon}&format=json&zoom=18"
    response = requests.get(url, headers={'User-Agent': 'TrafficPrediction/1.0'})
    if response.status_code == 200:
        data = response.json()
        address = data.get('address', {})

        # Prioritize shorter fields
        name = (
            address.get('road') or
            address.get('neighbourhood') or
            address.get('suburb') or
            address.get('district') or
            address.get('city') or
            address.get('town') or
            address.get('village') or
            address.get('county') or
            "Unknown"
        )

        # Truncate if still too long
        return name[:max_length]
    return f"{lat:.4f},{lon:.4f}"

def generate_route_name(start_lat, start_lon, end_lat, end_lon):
    """Generate short route name."""
    start = get_location_name(start_lat, start_lon, max_length=40)
    end = get_location_name(end_lat, end_lon, max_length=40)
    return f"{start} → {end}"

## 5.2 Update Route Names

In [ ]:
# @title
with engine.connect() as conn:
    for idx, row in routes_df.iterrows():
        route_name = generate_route_name(row['start_lat'], row['start_lon'], row['end_lat'], row['end_lon'])
        route_id = row['route_id']
        conn.execute(
            text("UPDATE routes SET route_name = :name WHERE route_id = :id"),
            {'name': route_name, 'id': route_id}
        )
        conn.commit()
        print(f"Updated route {route_id}: {route_name}")
    time.sleep(2)  # Nominatim API requires longer delay between calls

# 6. Route Geometries (TomTom Polylines)
## 6.1 TomTom API Functions

In [ ]:
# @title
def get_route_polyline(api_key, start_lat, start_lon, end_lat, end_lon):
    """Fetch route polyline from TomTom Routing API."""
    url = f"https://api.tomtom.com/routing/1/calculateRoute/{start_lat},{start_lon}:{end_lat},{end_lon}/json"
    params = {
        "key": api_key,
        "travelMode": "car",
        "traffic": "false",  # No need for traffic data for static geometry
        "routeType": "fastest"
    }

    try:
        response = requests.get(url, params=params, timeout=30)
        if response.status_code == 200:
            data = response.json()
            points = data['routes'][0]['legs'][0]['points']

            # Convert to GeoJSON LineString format [lon, lat]
            coordinates = [[point['longitude'], point['latitude']] for point in points]
            geojson = {
                "type": "LineString",
                "coordinates": coordinates
            }

            length_meters = data['routes'][0]['summary']['lengthInMeters']
            return geojson, length_meters
        else:
            print(f"  API Error {response.status_code}: {response.text[:100]}")
            return None, None
    except Exception as e:
        print(f"  Exception: {e}")
        return None, NO_CHANGE

## 6.2 Database Functions

In [ ]:
# @title
def route_geometry_exists(conn, route_id):
    """Check if route_geometries already has an entry for this route_id."""
    result = conn.execute(
        text("SELECT 1 FROM route_geometries WHERE route_id = :route_id"),
        {'route_id': route_id}
    ).fetchone()
    return result is not None

def insert_route_geometry(conn, route_id, geojson, length_meters, straightness_ratio):
    """Insert or update route geometry."""
    query = """
    INSERT INTO route_geometries (route_id, polyline_geojson, length_meters, straightness_ratio, updated_at)
    VALUES (:route_id, :geojson, :length_meters, :straightness_ratio, NOW())
    ON CONFLICT (route_id)
    DO UPDATE SET
        polyline_geojson = EXCLUDED.polyline_geojson,
        length_meters = EXCLUDED.length_meters,
        straightness_ratio = EXCLUDED.straightness_ratio,
        updated_at = NOW()
    """
    conn.execute(text(query), {
        'route_id': route_id,
        'geojson': json.dumps(geojson),
        'length_meters': length_meters,
        'straightness_ratio': straightness_ratio
    })
    conn.commit()

## 6.3 Helper Functions

In [ ]:
# @title
def haversine(lat1, lon1, lat2, lon2):
    """Calculate distance between two points in meters."""
    R = 6371000  # Earth radius in meters
    phi1 = radians(lat1)
    phi2 = radians(lat2)
    delta_phi = radians(lat2 - lat1)
    delta_lambda = radians(lon2 - lon1)

    a = sin(delta_phi/2)**2 + cos(phi1) * cos(phi2) * sin(delta_lambda/2)**2
    c = 2 * atan2(sqrt(a), sqrt(1-a))
    return R * c

def compute_straightness_ratio(coordinates, total_length_meters):
    """Calculate straightness ratio (1.0 = perfectly straight)."""
    if len(coordinates) < 2 or total_length_meters == 0:
        return 1.0

    # GeoJSON uses [lon, lat] order
    start_lon, start_lat = coordinates[0]
    end_lon, end_lat = coordinates[-1]

    straight_distance = haversine(start_lat, start_lon, end_lat, end_lon)
    return straight_distance / total_length_meters if total_length_meters > 0 else 1.0

## 6.4 Populate Route Geometries

In [ ]:
# @title
with engine.connect() as conn:
    for idx, row in routes_df.iterrows():
        route_id = row['route_id']
        print(f"Route {route_id:2}: ", end='')

        # Skip if already exists (optional: comment out to force refresh)
        if route_geometry_exists(conn, route_id):
            print(f"already exists, skipping")
            continue

        geojson, length = get_route_polyline(
            TOMTOM_API_KEY,
            row['start_lat'], row['start_lon'],
            row['end_lat'], row['end_lon']
        )

        if geojson and length:
            # Extract coordinates from GeoJSON for straightness calculation
            coordinates = geojson['coordinates']
            straightness = compute_straightness_ratio(coordinates, length)
            insert_route_geometry(conn, row['route_id'], geojson, length, straightness)
            print(f"length={length:5}m, straightness={straightness:.3f}")
        else:
            print("failed to fetch")

        # Be polite to TomTom API
        time.sleep(DELAY_BETWEEN_API_CALLS)